VIII) GOfuncR analysis of early human–chimpanzee PFC expression differences

Input: results/early_human_chimp_pylimma_results_human_spec.csv from the earlier pylimma analysis.

1.) Rank all tested genes by the signed pylimma moderated t-statistic. Positive values indicate human-higher expression; negative values indicate chimpanzee-higher expression. Use the GOfuncR Wilcoxon analysis.

2.) Test overrepresentation of the selected 140 genes by a hypergeometric test against the full tested-gene background.

3.) Correct GO term significance by permutation-based family-wise error rate (FWER). Use 10,000 random sets and FWER <= 0.05.

4.) Refine the significant GO terms within their hierarchy for each species direction.

The age cutoff in the preceding pylimma notebook was chronological age < 100 days. Species-adapted age was used as its regression covariate.


1.) Load R packages

Use the existing R kernel with GOfuncR and Homo.sapiens installed.


In [1]:
options(stringsAsFactors = FALSE)

suppressPackageStartupMessages({
    library(GOfuncR)
    library(Homo.sapiens)
})

dir.create("results", showWarnings = FALSE, recursive = TRUE)

cat("Environment ready for GO enrichment analysis!\n")

Environment ready for GO enrichment analysis!


2.) Settings

Use 10,000 random sets with random seed 42. Outputs have distinct filenames so the previous 10,000-random-set run is preserved.


In [2]:
RESULTS_FILE <- "results/early_human_chimp_pylimma_results_human_spec.csv"

N_RANDSETS <- 10000L
RANDOM_SEED <- 42L
FWER_CUTOFF <- 0.05
GO_DOMAIN <- "biological_process"
MAX_PLOT_TERMS <- 12L

RANKED_ALL_FILE <- "results/gofuncr_ranked_all_terms_human_spec.csv"
RANKED_SIGNIFICANT_FILE <- "results/gofuncr_ranked_significant_human_spec.csv"
RANKED_VALID_INPUT_FILE <- "results/gofuncr_ranked_valid_input_human_spec.csv"

SELECTED_ALL_FILE <- "results/gofuncr_selected_all_terms_human_spec.csv"
SELECTED_SIGNIFICANT_FILE <- "results/gofuncr_selected_significant_human_spec.csv"
SELECTED_VALID_INPUT_FILE <- "results/gofuncr_selected_valid_input_human_spec.csv"

GENE_INPUT_FILE <- "results/gofuncr_gene_inputs_human_spec.csv"
ANNOTATION_VERSION_FILE <- "results/gofuncr_annotation_versions_human_spec.csv"
SUMMARY_PLOT_FILE <- "results/gofuncr_go_summary_human_spec.png"

set.seed(RANDOM_SEED)

3.) Load pylimma results

Use the moderated t-statistic and the selected column from the saved Python results. Match genes using uppercase gene symbols.


In [3]:
if (!file.exists(RESULTS_FILE)) {
    stop(
        "Input file not found: ", RESULTS_FILE, "\n",
        "Copy the complete pylimma result CSV from the previous notebook into the data folder."
    )
}

pylimma_results <- read.csv(
    RESULTS_FILE,
    check.names = FALSE,
    stringsAsFactors = FALSE
)

required_columns <- c("Gene", "moderated_t_zero", "selected")
missing_columns <- setdiff(required_columns, names(pylimma_results))
if (length(missing_columns) > 0) {
    stop("Missing required columns: ", paste(missing_columns, collapse = ", "))
}

parse_logical_column <- function(x) {
    if (is.logical(x)) {
        return(x)
    }

    values <- tolower(trimws(as.character(x)))
    parsed <- rep(NA, length(values))
    parsed[values %in% c("true", "t", "1", "yes", "y")] <- TRUE
    parsed[values %in% c("false", "f", "0", "no", "n")] <- FALSE

    bad_values <- unique(values[!is.na(values) & is.na(parsed)])
    if (length(bad_values) > 0) {
        stop(
            "Could not interpret these values in the selected column: ",
            paste(bad_values, collapse = ", ")
        )
    }
    parsed
}

pylimma_results$Gene <- toupper(trimws(as.character(pylimma_results$Gene)))
pylimma_results$moderated_t_zero <- suppressWarnings(
    as.numeric(pylimma_results$moderated_t_zero)
)
pylimma_results$selected <- parse_logical_column(pylimma_results$selected)

valid_gene <- !is.na(pylimma_results$Gene) & nzchar(pylimma_results$Gene)
valid_score <- is.finite(pylimma_results$moderated_t_zero)
valid_selected <- !is.na(pylimma_results$selected)

if (any(!valid_gene | !valid_score | !valid_selected)) {
    message(
        "Removing ", sum(!valid_gene | !valid_score | !valid_selected),
        " rows with missing gene symbols, non-finite scores, or missing selected status."
    )
    pylimma_results <- pylimma_results[
        valid_gene & valid_score & valid_selected,
        ,
        drop = FALSE
    ]
}

if ("treat_p_value" %in% names(pylimma_results)) {
    duplicate_order <- order(
        pylimma_results$Gene,
        suppressWarnings(as.numeric(pylimma_results$treat_p_value)),
        -abs(pylimma_results$moderated_t_zero),
        na.last = TRUE
    )
} else {
    duplicate_order <- order(
        pylimma_results$Gene,
        -abs(pylimma_results$moderated_t_zero),
        na.last = TRUE
    )
}

pylimma_results <- pylimma_results[duplicate_order, , drop = FALSE]
duplicate_count <- sum(duplicated(pylimma_results$Gene))
if (duplicate_count > 0) {
    message("Removing ", duplicate_count, " duplicate gene-symbol rows.")
    pylimma_results <- pylimma_results[!duplicated(pylimma_results$Gene), , drop = FALSE]
}

rownames(pylimma_results) <- NULL

tested_genes <- pylimma_results$Gene
selected_genes <- pylimma_results$Gene[pylimma_results$selected]

gene_inputs <- data.frame(
    Gene = pylimma_results$Gene,
    moderated_t_zero = pylimma_results$moderated_t_zero,
    selected = pylimma_results$selected,
    rank_direction = ifelse(
        pylimma_results$moderated_t_zero > 0,
        "Human-higher",
        ifelse(
            pylimma_results$moderated_t_zero < 0,
            "Chimpanzee-higher",
            "No signed difference"
        )
    ),
    stringsAsFactors = FALSE
)

write.csv(gene_inputs, GENE_INPUT_FILE, row.names = FALSE)

cat("Unique genes in the tested universe:", length(tested_genes), "\n")
cat("Selected genes:", length(selected_genes), "\n")
cat("Selected Human-higher genes:", sum(
    pylimma_results$selected & pylimma_results$moderated_t_zero > 0
), "\n")
cat("Selected Chimpanzee-higher genes:", sum(
    pylimma_results$selected & pylimma_results$moderated_t_zero < 0
), "\n")

if (length(selected_genes) != 140L) {
    warning(
        "The current input contains ", length(selected_genes),
        " selected genes rather than the previously observed 140. ",
        "This is acceptable if the pylimma settings or input data changed."
    )
}

head(gene_inputs)

Unique genes in the tested universe: 11667 
Selected genes: 140 
Selected Human-higher genes: 62 
Selected Chimpanzee-higher genes: 78 


,Gene,moderated_t_zero,selected,rank_direction
,<chr>,<dbl>,<lgl>,<chr>
1,A2ML1,0.7281244,FALSE,Human-higher
2,AAAS,1.0525089,FALSE,Human-higher
3,AACS,-1.9647350,FALSE,Chimpanzee-higher
4,AADAT,-0.0625153,FALSE,Chimpanzee-higher
5,AAED1,0.8812522,FALSE,Human-higher
6,AAGAB,-2.3202451,FALSE,Chimpanzee-higher


4.) Ranked GO analysis

The Wilcoxon analysis uses all tested genes with GO annotations. High ranks are human-higher and low ranks are chimpanzee-higher.


In [4]:
input_ranked <- data.frame(
    gene_id = pylimma_results$Gene,
    score = pylimma_results$moderated_t_zero,
    stringsAsFactors = FALSE
)

set.seed(RANDOM_SEED)
ranked_go <- go_enrich(
    genes = input_ranked,
    test = "wilcoxon",
    n_randsets = N_RANDSETS,
    organismDb = "Homo.sapiens",
    domains = GO_DOMAIN,
    silent = FALSE
)

ranked_stats <- as.data.frame(ranked_go[[1]], stringsAsFactors = FALSE)
expected_ranked_columns <- c(
    "ontology", "node_id", "node_name",
    "raw_p_low_rank", "raw_p_high_rank",
    "FWER_low_rank", "FWER_high_rank"
)
missing_ranked_columns <- setdiff(expected_ranked_columns, names(ranked_stats))
if (length(missing_ranked_columns) > 0) {
    stop(
        "Unexpected GOfuncR ranked-result format. Missing: ",
        paste(missing_ranked_columns, collapse = ", ")
    )
}

ranked_stats$best_FWER <- pmin(
    ranked_stats$FWER_high_rank,
    ranked_stats$FWER_low_rank,
    na.rm = TRUE
)
ranked_stats$direction_of_smaller_FWER <- ifelse(
    ranked_stats$FWER_high_rank <= ranked_stats$FWER_low_rank,
    "Human-higher",
    "Chimpanzee-higher"
)
ranked_stats$significant_FWER_0_05 <- with(
    ranked_stats,
    FWER_high_rank <= FWER_CUTOFF | FWER_low_rank <= FWER_CUTOFF
)

ranked_stats <- ranked_stats[
    order(
        ranked_stats$best_FWER,
        pmin(ranked_stats$raw_p_high_rank, ranked_stats$raw_p_low_rank),
        ranked_stats$node_id
    ),
    ,
    drop = FALSE
]
rownames(ranked_stats) <- NULL

ranked_significant <- ranked_stats[
    ranked_stats$significant_FWER_0_05,
    ,
    drop = FALSE
]

write.csv(ranked_stats, RANKED_ALL_FILE, row.names = FALSE)
write.csv(ranked_significant, RANKED_SIGNIFICANT_FILE, row.names = FALSE)
write.csv(
    as.data.frame(ranked_go[[2]], stringsAsFactors = FALSE),
    RANKED_VALID_INPUT_FILE,
    row.names = FALSE
)

cat("Genes accepted by GOfuncR:", nrow(ranked_go[[2]]), "\n")
cat("Biological Process terms tested:", nrow(ranked_stats), "\n")
cat(
    "FWER-significant Human-higher terms:",
    sum(ranked_stats$FWER_high_rank <= FWER_CUTOFF), "\n"
)
cat(
    "FWER-significant Chimpanzee-higher terms:",
    sum(ranked_stats$FWER_low_rank <= FWER_CUTOFF), "\n"
)

ranked_display_columns <- c(
    "node_id", "node_name", "direction_of_smaller_FWER",
    "raw_p_high_rank", "FWER_high_rank",
    "raw_p_low_rank", "FWER_low_rank"
)
head(ranked_stats[, ranked_display_columns, drop = FALSE], 20)

Checking arguments...

Write temporary files...

Find associated categories using database 'Homo.sapiens'...

Remove annotated categories not present in GO-graph...

Found root_nodes: biological_process, cellular_component, molecular_function

Remove invalid genes...

Warning message in go_enrich(genes = input_ranked, test = "wilcoxon", n_randsets = N_RANDSETS, :
“No GO-annotation for genes: AAED1, AARS, ACRC, ACTA2-AS1, ADAL, ADAMTSL4-AS1, ADCK3, ADH5P4, ADPRHL2, ADRBK1, ADRBK2, ADSS, ADSSL1, AES, AFG3L1P, AGPAT4-IT1, AGPAT6, AGPAT9, AHSA2, ALMS1P, ALS2CR11, ALS2CR12, ANKRD32, ANP32C, APOA1BP, AQP4-AS1, ARNTL, ARNTL2, ARSE, ASB16-AS1, ASNA1, ASUN, ATP5A1, ATP5B, ATP5C1, ATP5D, ATP5EP2, ATP5F1, ATP5G1, ATP5G2, ATP5G3, ATP5H, ATP5I, ATP5J, ATP5L, ATP5L2, ATP5O, ATP5S, ATP5SL, ATP6AP1L, ATP6V0E2-AS1, ATPIF1, ATXN7L2, B3GALTL, B3GNT1, BAI1, BAI2, BAI3, BAIAP2-AS1, BHLHB9, BNIP3P1, BRD7P2, BRE, BTBD11, BTF3P11, BVES, BZRAP1, C10ORF10, C10ORF107, C10ORF113, C10ORF12, C10ORF2, C10ORF25, C10O

Read 47205 terms.
Found 28642 nodes.
Graph created.
Data and annotation file parsed.
Number of randomsets: 10000.
Computing randomsets...10000.
Finished

Evaluating randomsets: 
No. of significant ontology nodes for
low ranks				high ranks
of candidate genes at p-value thresholds
0.1	0.05	0.01	0.001	0.0001	0.1	0.05	0.01	0.001	0.0001
1269	482	50	2	0	1739	788	115	5	0	
1516	629	79	0	0	1503	650	86	3	0	
1180	475	54	2	1	1811	815	130	12	3	
1299	505	67	8	0	1797	786	116	8	1	
1633	707	82	11	1	1427	620	66	2	0	
1294	486	70	12	0	1613	753	124	11	1	
1497	601	71	5	0	1517	626	97	17	1	
1544	639	74	3	0	1330	556	74	4	0	
1810	874	145	6	0	1281	560	121	7	0	
1446	601	98	10	0	1407	607	88	11	3	
1391	609	81	4	0	1525	637	70	7	0	
1441	572	66	3	0	1546	701	95	6	0	
1433	673	92	6	0	1649	651	61	4	0	
1311	541	60	3	0	1450	561	77	6	0	
1585	649	103	11	1	1299	531	50	4	0	
1484	608	96	5	0	1528	667	96	4	0	
1508	633	58	2	0	1490	568	68	5	0	
1319	513	53	0	0	1533	679	97	5	0	
1208	470	71	5	0	1643	662	93	14	2	
1435	591	66	3	0	1645	7


Done.



Genes accepted by GOfuncR: 10386 
Biological Process terms tested: 13796 
FWER-significant Human-higher terms: 76 
FWER-significant Chimpanzee-higher terms: 106 


,node_id,node_name,direction_of_smaller_FWER,raw_p_high_rank,FWER_high_rank,raw_p_low_rank,FWER_low_rank
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,GO:0006139,nucleobase-containing compound metabolic process,Human-higher,0.000000e+00,0,1.000000e+00,1
2,GO:0006725,cellular aromatic compound metabolic process,Human-higher,0.000000e+00,0,1.000000e+00,1
3,GO:0010467,gene expression,Human-higher,0.000000e+00,0,1.000000e+00,1
4,GO:0016070,RNA metabolic process,Human-higher,0.000000e+00,0,1.000000e+00,1
5,GO:0034641,cellular nitrogen compound metabolic process,Human-higher,0.000000e+00,0,1.000000e+00,1
6,GO:0043170,macromolecule metabolic process,Human-higher,0.000000e+00,0,1.000000e+00,1
7,GO:0046483,heterocycle metabolic process,Human-higher,0.000000e+00,0,1.000000e+00,1
8,GO:0060255,regulation of macromolecule metabolic process,Human-higher,0.000000e+00,0,1.000000e+00,1
9,GO:0090304,nucleic acid metabolic process,Human-higher,0.000000e+00,0,1.000000e+00,1


5.) Selected-gene GO analysis

The hypergeometric test examines the selected gene set against all tested genes, restricted to those with GO annotations.


In [5]:
input_hyper <- data.frame(
    gene_id = pylimma_results$Gene,
    is_candidate = as.integer(pylimma_results$selected),
    stringsAsFactors = FALSE
)

if (sum(input_hyper$is_candidate) == 0) {
    stop("No selected genes are present in the input table.")
}

set.seed(RANDOM_SEED)
selected_go <- go_enrich(
    genes = input_hyper,
    test = "hyper",
    n_randsets = N_RANDSETS,
    organismDb = "Homo.sapiens",
    gene_len = FALSE,
    domains = GO_DOMAIN,
    silent = FALSE
)

selected_stats <- as.data.frame(selected_go[[1]], stringsAsFactors = FALSE)
expected_selected_columns <- c(
    "ontology", "node_id", "node_name",
    "raw_p_underrep", "raw_p_overrep",
    "FWER_underrep", "FWER_overrep"
)
missing_selected_columns <- setdiff(expected_selected_columns, names(selected_stats))
if (length(missing_selected_columns) > 0) {
    stop(
        "Unexpected GOfuncR hypergeometric-result format. Missing: ",
        paste(missing_selected_columns, collapse = ", ")
    )
}

selected_stats$significant_overrepresentation_FWER_0_05 <- (
    selected_stats$FWER_overrep <= FWER_CUTOFF
)
selected_stats <- selected_stats[
    order(
        selected_stats$FWER_overrep,
        selected_stats$raw_p_overrep,
        selected_stats$node_id
    ),
    ,
    drop = FALSE
]
rownames(selected_stats) <- NULL

selected_significant <- selected_stats[
    selected_stats$significant_overrepresentation_FWER_0_05,
    ,
    drop = FALSE
]

write.csv(selected_stats, SELECTED_ALL_FILE, row.names = FALSE)
write.csv(selected_significant, SELECTED_SIGNIFICANT_FILE, row.names = FALSE)
write.csv(
    as.data.frame(selected_go[[2]], stringsAsFactors = FALSE),
    SELECTED_VALID_INPUT_FILE,
    row.names = FALSE
)

valid_hyper_input <- as.data.frame(selected_go[[2]], stringsAsFactors = FALSE)
cat("Genes accepted by GOfuncR:", nrow(valid_hyper_input), "\n")
cat("Selected genes accepted by GOfuncR:", sum(valid_hyper_input[, 2] == 1), "\n")
cat("Biological Process terms tested:", nrow(selected_stats), "\n")
cat(
    "FWER-significant overrepresented terms:",
    nrow(selected_significant), "\n"
)

selected_display_columns <- c(
    "node_id", "node_name",
    "raw_p_overrep", "FWER_overrep",
    "raw_p_underrep", "FWER_underrep"
)
head(selected_stats[, selected_display_columns, drop = FALSE], 20)

Checking arguments...

Write temporary files...

Find associated categories using database 'Homo.sapiens'...

Remove annotated categories not present in GO-graph...

Found root_nodes: biological_process, cellular_component, molecular_function

Remove invalid genes...

Warning message in go_enrich(genes = input_hyper, test = "hyper", n_randsets = N_RANDSETS, :
“No GO-annotation for genes: AAED1, AARS, ACRC, ACTA2-AS1, ADAL, ADAMTSL4-AS1, ADCK3, ADH5P4, ADPRHL2, ADRBK1, ADRBK2, ADSS, ADSSL1, AES, AFG3L1P, AGPAT4-IT1, AGPAT6, AGPAT9, AHSA2, ALMS1P, ALS2CR11, ALS2CR12, ANKRD32, ANP32C, APOA1BP, AQP4-AS1, ARNTL, ARNTL2, ARSE, ASB16-AS1, ASNA1, ASUN, ATP5A1, ATP5B, ATP5C1, ATP5D, ATP5EP2, ATP5F1, ATP5G1, ATP5G2, ATP5G3, ATP5H, ATP5I, ATP5J, ATP5L, ATP5L2, ATP5O, ATP5S, ATP5SL, ATP6AP1L, ATP6V0E2-AS1, ATPIF1, ATXN7L2, B3GALTL, B3GNT1, BAI1, BAI2, BAI3, BAIAP2-AS1, BHLHB9, BNIP3P1, BRD7P2, BRE, BTBD11, BTF3P11, BVES, BZRAP1, C10ORF10, C10ORF107, C10ORF113, C10ORF12, C10ORF2, C10ORF25, C10ORF35

Read 47205 terms.
Found 28642 nodes.
Graph created.
Reading nodes_per_gene file... 
Found 9759 usable entrys in /tmp/RtmpQEBuPu/filecbb61b41f87f_GO:0008150 with 13796 GOs
Creating 10000 randomsets with size 104
Finished

Evaluating randomsets: 
No. of significant ontology nodes for
under-representation			over-representation
of candidate genes at p-value thresholds
0.1	0.05	0.01	0.001	0.0001	0.1	0.05	0.01	0.001	0.0001
16	4	1	1	0	484	224	11	1	0	
42	18	1	0	0	632	349	46	7	1	
59	19	5	0	0	400	198	17	3	1	
42	8	0	0	0	341	159	12	0	0	
10	2	0	0	0	563	291	27	2	0	
107	61	27	1	0	382	186	9	3	0	
84	34	5	0	0	251	106	13	1	0	
20	0	0	0	0	524	251	35	7	1	
62	27	7	0	0	435	231	27	8	0	
15	6	0	0	0	649	348	47	4	0	
25	11	1	0	0	457	244	24	4	0	
88	52	12	0	0	420	196	16	2	0	
36	13	1	0	0	495	248	18	0	0	
37	14	0	0	0	417	226	29	3	1	
15	0	0	0	0	658	292	21	3	0	
79	30	3	0	0	370	159	12	1	0	
75	49	15	3	0	448	235	24	7	2	
67	40	26	9	0	642	343	37	2	0	
64	30	4	0	0	632	337	51	7	0	
57	21	3	0	0	509	269	39	8	1	
88	25	6	0	0	312	165	1


Done.



Genes accepted by GOfuncR: 10386 
Selected genes accepted by GOfuncR: 116 
Biological Process terms tested: 13796 
FWER-significant overrepresented terms: 0 


,node_id,node_name,raw_p_overrep,FWER_overrep,raw_p_underrep,FWER_underrep
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,GO:0018916,nitrobenzene metabolic process,0.0003351107,0.5329,0.9999988,1
2,GO:0035106,operant conditioning,0.0006655542,0.7353,0.9999953,1
3,GO:0009264,deoxyribonucleotide catabolic process,0.0008540019,0.7720,0.9999668,1
4,GO:0046386,deoxyribose phosphate catabolic process,0.0008540019,0.7720,0.9999668,1
5,GO:0045143,homologous chromosome segregation,0.0022803024,0.9498,0.9998704,1
6,GO:0009166,nucleotide catabolic process,0.0027359922,0.9722,0.9997156,1
7,GO:0007094,mitotic spindle assembly checkpoint signaling,0.0031736488,0.9855,0.9997953,1
8,GO:0009394,2'-deoxyribonucleotide metabolic process,0.0031736488,0.9855,0.9997953,1
9,GO:0071173,spindle assembly checkpoint signaling,0.0031736488,0.9855,0.9997953,1


6.) Annotation versions

Record the GO annotation versions used for the analysis.


In [6]:
ranked_versions <- as.data.frame(ranked_go[[3]], stringsAsFactors = FALSE)
ranked_versions$analysis <- "all_gene_ranked_wilcoxon"

selected_versions <- as.data.frame(selected_go[[3]], stringsAsFactors = FALSE)
selected_versions$analysis <- "selected_set_hypergeometric"

annotation_versions <- rbind(ranked_versions, selected_versions)
annotation_versions <- annotation_versions[
    , c("analysis", setdiff(names(annotation_versions), "analysis")),
    drop = FALSE
]

write.csv(annotation_versions, ANNOTATION_VERSION_FILE, row.names = FALSE)
annotation_versions

analysis,type,db,version
<chr>,<chr>,<chr>,<chr>
all_gene_ranked_wilcoxon,go_annotations,Homo.sapiens,1.3.1
all_gene_ranked_wilcoxon,go_graph,integrated,23-Mar-2020
selected_set_hypergeometric,go_annotations,Homo.sapiens,1.3.1
selected_set_hypergeometric,go_graph,integrated,23-Mar-2020


7.) Summary

Report accepted genes, GO terms tested and the number significant at FWER <= 0.05.


In [7]:
analysis_summary <- data.frame(
    analysis = c(
        "Ranked: Human-higher",
        "Ranked: Chimpanzee-higher",
        "Selected set: overrepresented"
    ),
    genes_submitted = c(
        nrow(input_ranked),
        nrow(input_ranked),
        sum(input_hyper$is_candidate)
    ),
    genes_accepted_by_GOfuncR = c(
        nrow(ranked_go[[2]]),
        nrow(ranked_go[[2]]),
        sum(as.data.frame(selected_go[[2]])[, 2] == 1)
    ),
    terms_tested = c(
        nrow(ranked_stats),
        nrow(ranked_stats),
        nrow(selected_stats)
    ),
    significant_terms_FWER_0_05 = c(
        sum(ranked_stats$FWER_high_rank <= FWER_CUTOFF),
        sum(ranked_stats$FWER_low_rank <= FWER_CUTOFF),
        sum(selected_stats$FWER_overrep <= FWER_CUTOFF)
    ),
    stringsAsFactors = FALSE
)

analysis_summary

cat("\nOutput files written:\n")
cat(paste(
    c(
        RANKED_ALL_FILE,
        RANKED_SIGNIFICANT_FILE,
        RANKED_VALID_INPUT_FILE,
        SELECTED_ALL_FILE,
        SELECTED_SIGNIFICANT_FILE,
        SELECTED_VALID_INPUT_FILE,
        GENE_INPUT_FILE,
        ANNOTATION_VERSION_FILE
    ),
    collapse = "\n"
))
cat("\n")

analysis,genes_submitted,genes_accepted_by_GOfuncR,terms_tested,significant_terms_FWER_0_05
<chr>,<int>,<int>,<int>,<int>
Ranked: Human-higher,11667,10386,13796,76
Ranked: Chimpanzee-higher,11667,10386,13796,106
Selected set: overrepresented,140,116,13796,0



Output files written:
results/gofuncr_ranked_all_terms_human_spec.csv
results/gofuncr_ranked_significant_human_spec.csv
results/gofuncr_ranked_valid_input_human_spec.csv
results/gofuncr_selected_all_terms_human_spec.csv
results/gofuncr_selected_significant_human_spec.csv
results/gofuncr_selected_valid_input_human_spec.csv
results/gofuncr_gene_inputs_human_spec.csv
results/gofuncr_annotation_versions_human_spec.csv


8.) Reproducibility

Record the R session and package versions.


In [8]:
sessionInfo()

R version 4.5.3 (2026-03-11)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 24.04.5 LTS

Matrix products: default
BLAS/LAPACK: /home/bjoern/miniforge3/envs/testenv/lib/libopenblasp-r0.3.34.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: America/New_York
tzcode source: system (glibc)

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] Homo.sapiens_1.3.1                      
 [2] TxDb.Hsapiens.UCSC.hg19.knownGene_3.22.1
 [3] org.Hs.eg.db_3.22.0                     
 [4] GO.db_3.22.0                            
 [5] OrganismDbi_1.52.0        

9.) Refine GO categories

Remove contributions from significant child terms to identify more specific GO categories. Report only rows with signif = TRUE.



In [9]:
# Save the current 10,000-random-set analysis for reproducibility.
saveRDS(ranked_go, "results/gofuncr_ranked_complete.rds")

refined_human <- GOfuncR::refine(
    res = ranked_go,
    fwer = FWER_CUTOFF,
    fwer_col = "FWER_high_rank"
)
refined_chimp <- GOfuncR::refine(
    res = ranked_go,
    fwer = FWER_CUTOFF,
    fwer_col = "FWER_low_rank"
)

# The signif column marks terms that remain significant after refinement.
human_terms <- refined_human[!is.na(refined_human$signif) & refined_human$signif, , drop = FALSE]
chimp_terms <- refined_chimp[!is.na(refined_chimp$signif) & refined_chimp$signif, , drop = FALSE]

# Keep the full refinement outputs and save separate tables of retained terms.
write.csv(refined_human, "results/gofuncr_refined_human.csv", row.names = FALSE)
write.csv(refined_chimp, "results/gofuncr_refined_chimp.csv", row.names = FALSE)
write.csv(human_terms, "results/gofuncr_refined_human_significant.csv", row.names = FALSE)
write.csv(chimp_terms, "results/gofuncr_refined_chimp_significant.csv", row.names = FALSE)

# Save the GO summary plot advertised by the output settings.
png(filename = SUMMARY_PLOT_FILE, width = 1400, height = 800, res = 170, bg = "white")
barplot(
    height = c("Human-higher" = nrow(human_terms),
               "Chimpanzee-higher" = nrow(chimp_terms)),
    main = "Refined significant GO Biological Process terms",
    ylab = "Number of terms",
    ylim = c(0, max(nrow(human_terms), nrow(chimp_terms)) * 1.18),
    col = "gray75",
    border = "gray40"
)
dev.off()

cat("Human-higher categories after refinement:", nrow(human_terms), "\n")
cat("Chimpanzee-higher categories after refinement:", nrow(chimp_terms), "\n")

# Display all retained GO identifiers and names, with original FWER and refined p-value.
cat("\nHuman-higher refined categories:\n")
print(human_terms[, c("node_id", "node_name", "FWER_high_rank", "refined_p_high_rank"), drop = FALSE], row.names = FALSE)

cat("\nChimpanzee-higher refined categories:\n")
print(chimp_terms[, c("node_id", "node_name", "FWER_low_rank", "refined_p_low_rank"), drop = FALSE], row.names = FALSE)

cat("Saved GO summary plot:", SUMMARY_PLOT_FILE, "\n")


Found 76 significant categories at FWER threshold 0.05


Get annotations for categories:

find child nodes of GO-categories...

find genes annotated to child nodes using database 'Homo.sapiens'...

Remove annotated categories not present in GO-graph...

Done.


Get annotations for root nodes:

find child nodes of GO-categories...

find genes annotated to child nodes using database 'Homo.sapiens'...

Remove annotated categories not present in GO-graph...

Done.


Refine root node biological_process

using p-value threshold 2.0870107208415e-05...


Found 8 leaves...

8 of the 8 leaves stay significant after refinement.

Refine next level...

Found 10 leaves...

0 of the 10 leaves stay significant after refinement.

Refine next level...

Found 9 leaves...

1 of the 9 leaves stay significant after refinement.

Refine next level...

Found 7 leaves...

0 of the 7 leaves stay significant after refinement.

Refine next level...

Found 9 leaves...

0 of the 9 leaves stay significant after refin

pdf 
  2

Human-higher categories after refinement: 10 
Chimpanzee-higher categories after refinement: 30 

Human-higher refined categories:
    node_id                                                      node_name
 GO:0019219 regulation of nucleobase-containing compound metabolic process
 GO:2000112      regulation of cellular macromolecule biosynthetic process
 GO:0006357               regulation of transcription by RNA polymerase II
 GO:0000398                                 mRNA splicing, via spliceosome
 GO:0019827                               stem cell population maintenance
 GO:0006974                       cellular response to DNA damage stimulus
 GO:0006259                                          DNA metabolic process
 GO:0098813                                 nuclear chromosome segregation
 GO:0045892            negative regulation of transcription, DNA-templated
 GO:0006323                                                  DNA packaging
 FWER_high_rank refined_p_high_rank
        